# UMLS Graph Exploration for Clinical Knowledge Injection

This notebook helps you explore the connectivity and semantic organization of the clinical variables used in the MIMIC-IV experiments. We use it to verify which concepts (CUIs) are available and how they are linked in the UMLS knowledge graph.

In [1]:
import pandas as pd
import numpy as np
import os
from tqdm.auto import tqdm

# Paths to UMLS RRF files
UMLS_DIR = "/home/guido/Data/indus_data/ontologies/2025AB/META"
MRREL_PATH = os.path.join(UMLS_DIR, "MRREL.RRF")
MRSTY_PATH = os.path.join(UMLS_DIR, "MRSTY.RRF")
MRCONSO_PATH = os.path.join(UMLS_DIR, "MRCONSO.RRF")

# Our 17 Target Variables
MIMIC_TARGETS = {
    "C0018810": "Heart Rate",
    "C0871470": "Systolic Blood Pressure",
    "C0428883": "Diastolic Blood Pressure",
    "C0231832": "Respiratory Rate",
    "C0483415": "Oxygen Saturation",
    "C0039476": "Temperature",
    "C0017725": "Glucose",
    "C0032821": "Potassium",
    "C0037473": "Sodium",
    "C0008203": "Chloride",
    "C0010294": "Creatinine",
    "C0005845": "BUN",
    "C0023516": "White Blood Cells",
    "C0005821": "Platelets",
    "C0019046": "Hemoglobin",
    "C0018935": "Hematocrit",
    "C0376261": "Lactate"
}

/home/guido/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Analyze Semantic Types (TUIs)

This cell checks what "Semantic Type" each variable has in UMLS. **Crucial point:** If our pruning logic filters out these types, these variables will have zero edges to other clinical concepts.

In [2]:
tui_map = {
    "T201": "Clinical Attribute",
    "T059": "Laboratory Procedure",
    "T116": "Amino Acid, Peptide, or Protein",
    "T121": "Pharmacologic Substance",
    "T123": "Biologically Active Substance",
    "T196": "Element, Ion, or Isotope",
    "T081": "Quantitative Concept",
    "T109": "Organic Chemical",
    "T025": "Cell"
}

results = []
target_cuis = set(MIMIC_TARGETS.keys())
with open(MRSTY_PATH, 'r') as f:
    for line in f:
        parts = line.split('|')
        if parts[0] in target_cuis:
            results.append({'CUI': parts[0], 'Label': MIMIC_TARGETS[parts[0]], 'TUI': parts[1]})

df_tui = pd.DataFrame(results)
df_tui['Semantic Type Name'] = df_tui['TUI'].map(lambda x: tui_map.get(x, x))
display(df_tui.sort_values('Label'))

print("\nSummary of Semantic Types found in our targets:")
print(df_tui['TUI'].value_counts())

,CUI,Label,TUI,Semantic Type Name
1,C0005845,BUN,T059,Laboratory Procedure
2,C0008203,Chloride,T196,"Element, Ion, or Isotope"
3,C0010294,Creatinine,T109,Organic Chemical
4,C0010294,Creatinine,T123,Biologically Active Substance
22,C0428883,Diastolic Blood Pressure,T201,Clinical Attribute
5,C0017725,Glucose,T109,Organic Chemical
6,C0017725,Glucose,T121,Pharmacologic Substance
7,C0017725,Glucose,T123,Biologically Active Substance
8,C0018810,Heart Rate,T201,Clinical Attribute
9,C0018935,Hematocrit,T059,Laboratory Procedure



Summary of Semantic Types found in our targets:
TUI
T123    5
T201    5
T196    3
T109    3
T121    3
T025    2
T059    2
T116    1
T081    1
Name: count, dtype: int64


## 2. Check Raw Edge Counts (Connectivity)

Even before pruning, how many edges do these concepts have in the full UMLS (`MRREL`)?

In [3]:
counts = {cui: 0 for cui in target_cuis}
rel_types = {cui: set() for cui in target_cuis}
edge_sample = []

with open(MRREL_PATH, 'r') as f:
    for line in tqdm(f, desc="Scanning MRREL (60M+ lines)"):
        parts = line.split('|')
        c1, c2, rel, rela = parts[0], parts[4], parts[3], parts[7]
        if c1 in target_cuis:
            counts[c1] += 1
            rel_types[c1].add(rel)
        if c2 in target_cuis:
            counts[c2] += 1
            rel_types[c2].add(rel)

summary = []
for cui, label in MIMIC_TARGETS.items():
    summary.append({
        'Label': label,
        'Total Edges': counts[cui],
        'Rel Types': ', '.join(list(rel_types[cui])[:5])
    })
df_conn = pd.DataFrame(summary).sort_values('Total Edges', ascending=False)
display(df_conn)

Scanning MRREL (60M+ lines): 63494934it [02:08, 495350.78it/s]


,Label,Total Edges,Rel Types
6,Glucose,14502,"QB, RN, AQ, RO, RB"
12,White Blood Cells,9964,"QB, RN, AQ, RO, RB"
10,Creatinine,8630,"QB, RN, AQ, RO, RB"
7,Potassium,3970,"QB, RN, AQ, RO, RB"
8,Sodium,3488,"QB, RN, AQ, RO, RB"
13,Platelets,3388,"QB, RN, AQ, RO, RB"
14,Hemoglobin,2846,"QB, RN, AQ, RO, RB"
0,Heart Rate,2174,"QB, RN, AQ, RO, RB"
15,Hematocrit,1590,"QB, RN, AQ, RO, RB"
11,BUN,1390,"RN, RO, RB, PAR, CHD"


## 3. Verify Specific Path: Heart Rate to Glucose?

Let's see if there is any direct or 2-hop relationship between two conceptually different variables.

In [4]:
def find_neighbors(target_cui, max_edges=10):
    neighbors = []
    with open(MRREL_PATH, 'r') as f:
        for line in f:
            parts = line.split('|')
            c1, c2, rel, rela = parts[0], parts[4], parts[3], parts[7]
            if c1 == target_cui:
                neighbors.append({'Neighbor': c2, 'REL': rel, 'RELA': rela})
            elif c2 == target_cui:
                neighbors.append({'Neighbor': c1, 'REL': rel, 'RELA': rela})
            if len(neighbors) >= max_edges: break
    return pd.DataFrame(neighbors)

print("Sample neighbors for Heart Rate (C0018810):")
display(find_neighbors("C0018810"))

Sample neighbors for Heart Rate (C0018810):


,Neighbor,REL,RELA
0,C0004238,RO,is_interpreted_by
1,C0004238,RO,is_interpreted_by
2,C0004239,RO,is_interpreted_by
3,C0004239,RO,is_interpreted_by
4,C0017399,QB,
5,C0018810,SY,permuted_term_of
6,C0018810,SY,permuted_term_of
7,C0018810,SY,translation_of
8,C0018810,SY,translation_of
9,C0018810,SY,translation_of
